In [ ]:
import sys
import os
from pathlib import Path  # noqa: F401

# Resolve project root regardless of where the notebook is launched from
for _candidate in [".", "..", "../.."]:
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import matplotlib.pyplot as plt  # noqa: E402
import seaborn as sns  # noqa: E402
import mne  # noqa: E402

from src.preprocessing.pipeline import DatasetHandler  # noqa: E402
from src.definitions.fields import (  # noqa: E402
    ExperimentNames,
    CoordinateSystems,
    PreprocessedDataVariants,
    SingleDataMetadata,
    ConditionVariants,
    MusicTypeVariants,
)
from src.definitions.constants import ProjectPaths  # noqa: E402
from src.preprocessing.stimulus_alignment import (  # noqa: E402
    DEFAULT_STIMULUS_LABEL,
    get_stimulus_onset_samples,
)

%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)
mne.set_log_level("ERROR")
print("Setup complete.")

# ASSR Annotation Discrepancy — `.evt` Ground Truth vs What MNE Reads

Every onset-locked ASSR analysis is timed by the `fam+` annotations read out of the
raw EDF with MNE. Those markers are known to be wrong: the 40 Hz driven response
sits at **negative** times relative to them (see
[`assr_stimulus_onset_offset.ipynb`](assr_stimulus_onset_offset.ipynb), which
measured the lag at ≈ −380 ms from inter-trial phase coherence and froze it into
`AssrEpoch.MARKER_ONSET_OFFSET_S`, at the time a single global `-0.4` s).

That measurement said *how far off* the markers are on average. It could not say
**why**, nor whether the error is the same in every recording — an ITC fit has tens
of milliseconds of resolution at best, so a per-recording spread of that size would
have been invisible to it. This notebook answers both questions from an independent,
exact source.

**The independent source.** `data/events/` holds the recording-native `.evt` export
for all 38 ASSR recordings: every event with its time in **microseconds**, in the
acquisition software's own time base. It never passed through EDF, so it is the
reference against which the MNE read-out can be checked event by event.

**Question.** Is the discrepancy between the `.evt` times and the MNE annotation
times *systematic* (a recoverable, deterministic shift) or *random* (per-event
jitter that no correction can undo)?

**Steps.**
1. Anatomy of the discrepancy in one recording — `.evt` vs MNE, event by event.
2. Is the shift constant *within* a recording?
3. Is it constant *across* recordings? (scan of all 38)
4. Mechanism — where the shift comes from, and reconstruction from the `.evt`
   wall-clock anchor.
5. Does it survive our preprocessing? (raw EDF → after ICA → cropped → concatenated,
   the array the PCA analyses consume)
6. Consequences for `AssrEpoch.MARKER_ONSET_OFFSET_S` and what a per-recording
   correction would buy.

**Result obtained when this notebook was written** — the discrepancy is *fully*
systematic and exactly recoverable. Within a recording the MNE markers are one
constant shift late (residual spread ≤ 2 µs over ~150 markers, i.e. nothing but the
quantisation of the two time grids). Across recordings the shift is a per-recording
constant of **372–455 ms** (median 425 ms), which equals — to floating-point
precision — the offset between the two files' time origins: the EDF header can only
record the recording start to the nearest **whole second**, and the sub-second
remainder is recoverable only from the `.evt` wall-clock anchor. Nothing in our own
preprocessing adds to it.

## Configuration

In [ ]:
# ── Experiment under test ────────────────────────────────────────────────────
EXPERIMENT = ExperimentNames.ASSR
CONDITION = ConditionVariants.PLACEBO  # used for the pipeline-propagation check
MUSIC_TYPE = MusicTypeVariants.ASSR

STIMULUS_LABEL = DEFAULT_STIMULUS_LABEL  # "fam+" — the stimulus marker under test
CONTEXT_LABEL = "bgin"  # the companion marker, carried along as a cross-check

# The discrepancy is a property of how each *file* was written, not of the drug
# condition, so the scan covers every recording that has an `.evt` export. The
# condition only matters in step 5, which follows one group down the pipeline.
SCAN_ALL_RECORDINGS = True

# ── Reference event times ────────────────────────────────────────────────────
EVENTS_DIR = ProjectPaths.EVENTS_DIR  # `.evt` exports, one per recording
# `.evt` rows whose `TriNo` field parses as an ISO timestamp are the acquisition
# software's wall-clock anchors: they tie one microsecond time to absolute time,
# which is what makes the two files' time origins comparable in step 4.
EVT_TIME_COLUMN = "Tmu"  # event time, microseconds, `.evt` time base
EVT_LABEL_COLUMN = "Comnt"  # event label ("fam+", "bgin", ...)
EVT_TRIGGER_COLUMN = "TriNo"  # holds the ISO wall clock on anchor rows

# Tolerance below which a per-event residual counts as microsecond rounding
# rather than real jitter. The `.evt` grid is 1 µs and the EDF annotation grid is
# 1 ms, so anything at or under a few µs cannot be a genuine timing difference.
JITTER_TOLERANCE_S = 5e-6

# ── Example recording for the per-event walkthrough (steps 1-2) ──────────────
# None = the first recording of the scanned set.
EXAMPLE_FILE = None
N_EVENTS_SHOWN = 8  # rows printed in the side-by-side comparison

# ── Pipeline stages followed in step 5 ───────────────────────────────────────
STAGE_VARIANTS = [
    PreprocessedDataVariants.RAW_AFTER_ICA,
    PreprocessedDataVariants.RAW_CROPPED,
]
CONCAT_LABEL = f"{CONDITION.value}_{MUSIC_TYPE.value}"
CONCAT_SFREQ = 250.0  # rate the concatenated array was resampled to

# The single global offset the pipeline applied when this notebook was written,
# and whose adequacy step 6 tests. Pinned rather than read from `AssrEpoch`: this
# notebook is the diagnosis that motivated replacing that constant with a
# per-recording calibration, so tracking the live value would erase its own
# evidence. The outputs below were produced with exactly this value.
PIPELINE_CONSTANT_S = -0.400

# FIF stores annotation onsets in single precision, so two stages that did not
# re-time anything can still differ by ~10 µs. Below this, "unchanged" is the only
# reading; above it, a stage really moved the markers.
STAGE_TOLERANCE_S = 1e-4

# ── Plot saving ──────────────────────────────────────────────────────────────
SAVE_PLOTS = True
PLOTS_DIR = (
    ProjectPaths.NOTEBOOKS_DIR
    / "00-preprocessing"
    / "plots"
    / "assr_annotation_discrepancy"
)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Experiment    : {EXPERIMENT.value}")
print(f"Markers       : {STIMULUS_LABEL!r} (+ {CONTEXT_LABEL!r} as cross-check)")
print(f"Event exports : {EVENTS_DIR}")
print(f"Pipeline const: {PIPELINE_CONSTANT_S:+.3f} s (global, under test)")
print(f"Plots         : {PLOTS_DIR if SAVE_PLOTS else 'not saved'}")

## Helper Functions

Three small readers, one per source of truth: the `.evt` export, its wall-clock
anchor, and the MNE annotations. Keeping them separate is the point — the whole
notebook rests on the two time bases never being silently mixed.

Two details in `evt_marker_times` are load-bearing:

* **Duplicate rows are collapsed.** Some exports list the same event twice, once
  per hardware trigger line (e.g. `Code 2 / TriNo 0` and `Code 1 / TriNo 601` at
  an identical microsecond). Those are one event written twice, not two events;
  counting them would corrupt the index-wise pairing against the MNE annotations.
* **Times stay in the `.evt` frame.** No offset is applied here. Anything that
  looks like a correction belongs in step 4, after the mechanism is established.

In [ ]:
def read_evt_file(path: Path) -> pd.DataFrame:
    """Read a recording-native `.evt` event export.

    :param path: Path to the `.evt` file (tab-separated, one header row).
    :return: DataFrame with the original columns stripped of padding, plus
        ``time_s`` — the event time in seconds in the `.evt` time base.
    """
    events = pd.read_csv(path, sep="\t", header=0, dtype=str)
    events.columns = [column.strip() for column in events.columns]
    for column in events.columns:
        events[column] = events[column].astype(str).str.strip()
    events["time_s"] = events[EVT_TIME_COLUMN].astype(np.int64) / 1e6
    return events


def evt_marker_times(events: pd.DataFrame, label: str) -> np.ndarray:
    """Sorted, de-duplicated times of one marker label in the `.evt` time base.

    :param events: DataFrame from :func:`read_evt_file`.
    :param label: Marker label (case-insensitive), e.g. ``"fam+"``.
    :return: Sorted array of unique event times in seconds.
    """
    mask = events[EVT_LABEL_COLUMN].str.lower() == label.strip().lower()
    return np.unique(events.loc[mask, "time_s"].to_numpy())


def evt_duplicate_count(events: pd.DataFrame, label: str) -> int:
    """How many rows of ``label`` are repeats of a time already listed.

    :param events: DataFrame from :func:`read_evt_file`.
    :param label: Marker label (case-insensitive).
    :return: Number of surplus rows (0 when every event is listed once).
    """
    mask = events[EVT_LABEL_COLUMN].str.lower() == label.strip().lower()
    return int(mask.sum()) - len(evt_marker_times(events, label))


def evt_clock_anchor(events: pd.DataFrame) -> tuple[float, pd.Timestamp]:
    """The `.evt` wall-clock anchor: one event time tied to absolute time.

    Anchor rows are those whose trigger field holds an ISO timestamp instead of a
    numeric code. They are what makes the `.evt` time base comparable to the EDF
    header start time.

    :param events: DataFrame from :func:`read_evt_file`.
    :return: ``(time_s, wall_clock)`` of the first anchor row.
    :raises ValueError: If the export contains no anchor row.
    """
    stamps = pd.to_datetime(events[EVT_TRIGGER_COLUMN], errors="coerce", format="ISO8601")
    anchors = np.flatnonzero(stamps.notna().to_numpy())
    if not len(anchors):
        raise ValueError("No wall-clock anchor row in the `.evt` export.")
    row = anchors[0]
    return float(events["time_s"].iloc[row]), stamps.iloc[row]


def mne_marker_times(raw: mne.io.Raw, label: str) -> np.ndarray:
    """Sorted annotation times of one label, as MNE reads them.

    Returned in MNE's own frame: seconds from ``raw.annotations.orig_time``,
    i.e. from the recording start written in the file header. No offset and no
    ``first_time`` bookkeeping is applied — this is the number every onset-locked
    analysis ultimately consumes.

    :param raw: Recording whose annotations are read.
    :param label: Annotation description (case-insensitive).
    :return: Sorted array of annotation onsets in seconds.
    """
    target = label.strip().lower()
    return np.sort(
        np.array(
            [
                onset
                for onset, description in zip(
                    raw.annotations.onset, raw.annotations.description
                )
                if description.strip().lower() == target
            ]
        )
    )


def event_file_path(filename: str) -> Path:
    """`.evt` export belonging to a raw data filename (same stem).

    :param filename: Raw data filename, with or without extension.
    :return: Path to the matching `.evt` file.
    """
    return EVENTS_DIR / (Path(filename).stem + ".evt")

## Dataset Selection

In [ ]:
dataset_handler = DatasetHandler(EXPERIMENT, CoordinateSystems.HYDROGEL_257_NO_FIDUCIALS)
metadata = dataset_handler.dataset_metadata

group_df = metadata[
    (metadata[SingleDataMetadata.CONDITION] == CONDITION)
    & (metadata[SingleDataMetadata.MUSIC_TYPE] == MUSIC_TYPE)
]
scan_df = metadata if SCAN_ALL_RECORDINGS else group_df

# Only recordings that actually have an `.evt` export can be checked.
scan_files = [
    filename
    for filename in scan_df[SingleDataMetadata.FILENAME]
    if event_file_path(filename).exists()
]
missing = len(scan_df) - len(scan_files)

example_file = EXAMPLE_FILE or scan_files[0]

print(f"{EXPERIMENT.value}: {len(metadata)} recordings "
      f"({len(group_df)} in {CONDITION.value}/{MUSIC_TYPE.value})")
print(f"Scanned      : {len(scan_files)} with an `.evt` export"
      f"{f' ({missing} without — skipped)' if missing else ''}")
print(f"Example      : {example_file}")

## 1. Anatomy of the Discrepancy in One Recording

Read the same stimulus markers from both sources and put them side by side. Both
columns are "time from the start of the recording" as their own file defines it,
so if the two files agreed the difference column would be zero.

In [ ]:
example_events = read_evt_file(event_file_path(example_file))
example_raw = dataset_handler.load_data_file(example_file, preload=False)

evt_times = evt_marker_times(example_events, STIMULUS_LABEL)
mne_times = mne_marker_times(example_raw, STIMULUS_LABEL)

print(f"{example_file}")
print(f"  sfreq            : {example_raw.info['sfreq']:.0f} Hz, "
      f"{example_raw.n_times} samples ({example_raw.times[-1]:.1f} s)")
print(f"  EDF header start : {example_raw.info['meas_date']}")
print(f"  {STIMULUS_LABEL!r} in `.evt`  : {len(evt_times)}"
      f"  (+{evt_duplicate_count(example_events, STIMULUS_LABEL)} duplicate rows)")
print(f"  {STIMULUS_LABEL!r} in MNE     : {len(mne_times)}")

n_paired = min(len(evt_times), len(mne_times))
comparison = pd.DataFrame(
    {
        "evt_s": evt_times[:n_paired],
        "mne_s": mne_times[:n_paired],
        "difference_ms": (mne_times[:n_paired] - evt_times[:n_paired]) * 1e3,
    }
)
print()
print(comparison.head(N_EVENTS_SHOWN).to_string(float_format=lambda v: f"{v:10.4f}"))

The counts match exactly and the events line up one-to-one — **no marker is lost,
gained or reordered**. Every marker is simply read *late* by MNE, and by the same
amount. That already rules out the failure modes that would be unfixable (dropped
triggers, a resampling drift, a per-event race); what is left is a shift.

## 2. Is the Shift Constant *Within* a Recording?

A constant shift and a slow clock drift look identical in the table above — both
give a difference of "about 420 ms" on the first few events. They separate on the
*residual*: subtract the median difference and look at what is left over the full
~4-minute recording. A drift walks; a constant shift leaves nothing but the
quantisation of the two time grids.

In [ ]:
differences = mne_times[:n_paired] - evt_times[:n_paired]
residuals = differences - np.median(differences)

print(f"Difference : median {np.median(differences) * 1e3:8.3f} ms   "
      f"min {differences.min() * 1e3:8.3f} ms   max {differences.max() * 1e3:8.3f} ms")
print(f"Residual   : peak-to-peak {np.ptp(residuals) * 1e6:.3f} µs   "
      f"std {residuals.std() * 1e6:.3f} µs")
print(f"Tolerance  : {JITTER_TOLERANCE_S * 1e6:.0f} µs "
      f"(1 ms EDF annotation grid vs 1 µs `.evt` grid)")
print()
verdict = "CONSTANT" if np.ptp(residuals) <= JITTER_TOLERANCE_S else "NOT constant"
print(f"=> the shift is {verdict} within this recording.")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(evt_times, differences * 1e3, ".", ms=5)
# Absolute milliseconds on a ±1 ms window: an auto-scaled axis would show only the
# offset notation, which reads as noise rather than as "flat at 431 ms".
axes[0].ticklabel_format(axis="y", useOffset=False, style="plain")
axes[0].set_ylim((np.median(differences) - 1e-3) * 1e3,
                 (np.median(differences) + 1e-3) * 1e3)
axes[0].set(
    xlabel="event time in the `.evt` frame (s)",
    ylabel="MNE − `.evt` (ms)",
    title="Per-event discrepancy over the recording",
)
axes[1].plot(evt_times, residuals * 1e6, ".", ms=5)
axes[1].axhline(0.0, color="k", lw=0.8)
axes[1].set(
    xlabel="event time in the `.evt` frame (s)",
    ylabel="residual after removing the median (µs)",
    title="Residual — flat means one constant shift, not a drift",
)
fig.suptitle(f"{example_file}  —  {STIMULUS_LABEL!r} markers", y=1.02)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "per_event_discrepancy.png", dpi=150, bbox_inches="tight")
plt.show()

The residual is flat within a couple of **microseconds** — some 500× finer than the
1 ms grid the EDF annotations are written on, and exactly what one tick of each time
grid predicts. There is no drift and no per-event jitter: within a recording this is
a *single number*.

## 3. Is It Constant *Across* Recordings?

One shift per recording is only useful if that shift is predictable. Scan every
recording: extract the per-recording shift, its internal residual (step 2 repeated
for all files), and the wall-clock anchor that step 4 will need. `bgin` is carried
along as a cross-check — it is a different marker written by the same machinery, so
it must show the same shift if the cause is the file and not the stimulus code.

In [ ]:
scan_rows = []
for filename in scan_files:
    events = read_evt_file(event_file_path(filename))
    raw = dataset_handler.load_data_file(filename, preload=False)

    anchor_time_s, anchor_clock = evt_clock_anchor(events)
    header_start = pd.Timestamp(raw.info["meas_date"].replace(tzinfo=None))
    # Where the `.evt` time origin sits in the EDF's own time base.
    origin_offset_s = (anchor_clock - pd.Timedelta(seconds=anchor_time_s)
                       - header_start).total_seconds()

    row = {
        "file": Path(filename).stem,
        "header_start": header_start,
        "evt_origin_offset_s": origin_offset_s,
    }
    for label, key in [(STIMULUS_LABEL, "stimulus"), (CONTEXT_LABEL, "context")]:
        evt_label_times = evt_marker_times(events, label)
        mne_label_times = mne_marker_times(raw, label)
        row[f"n_evt_{key}"] = len(evt_label_times)
        row[f"n_mne_{key}"] = len(mne_label_times)
        row[f"n_dup_{key}"] = evt_duplicate_count(events, label)
        if len(evt_label_times) == len(mne_label_times) and len(evt_label_times):
            delta = mne_label_times - evt_label_times
            row[f"shift_{key}_s"] = float(np.median(delta))
            row[f"jitter_{key}_s"] = float(np.ptp(delta))
        else:
            row[f"shift_{key}_s"] = np.nan
            row[f"jitter_{key}_s"] = np.nan
    scan_rows.append(row)

scan = pd.DataFrame(scan_rows)
scan["counts_match"] = scan["n_evt_stimulus"] == scan["n_mne_stimulus"]
print(f"Scanned {len(scan)} recordings.")
scan[
    ["file", "n_evt_stimulus", "n_mne_stimulus", "n_dup_stimulus",
     "shift_stimulus_s", "jitter_stimulus_s", "shift_context_s"]
]

In [ ]:
paired = scan[scan["counts_match"]]
shifts = paired["shift_stimulus_s"].to_numpy()

print(f"Stimulus counts agree in {len(paired)}/{len(scan)} recordings.")
print()
print("Per-recording shift (MNE − `.evt`), seconds")
print(f"  min {shifts.min():.6f}   median {np.median(shifts):.6f}   "
      f"max {shifts.max():.6f}   spread {np.ptp(shifts) * 1e3:.1f} ms")
print(f"  IQR [{np.percentile(shifts, 25):.4f}, {np.percentile(shifts, 75):.4f}]")
print()
print(f"Worst within-recording jitter : "
      f"{paired['jitter_stimulus_s'].max() * 1e6:.3f} µs "
      f"(tolerance {JITTER_TOLERANCE_S * 1e6:.0f} µs)")

comparable_context = paired["shift_context_s"].notna()
context_agreement = (
    paired.loc[comparable_context, "shift_context_s"]
    - paired.loc[comparable_context, "shift_stimulus_s"]
).abs().max()
print(f"{CONTEXT_LABEL!r} vs {STIMULUS_LABEL!r} shift, max difference : "
      f"{context_agreement * 1e6:.3f} µs over "
      f"{int(comparable_context.sum())}/{len(paired)} comparable recordings"
      f"  -> the shift belongs to the file, not the marker")
print()

# Count-level anomalies. None of these are timing problems, but they are the
# reason the pairing above needs de-duplication and a `counts_match` guard.
anomalies = scan[
    (scan["n_dup_stimulus"] > 0)
    | (scan["n_dup_context"] > 0)
    | (scan["n_evt_context"] != scan["n_mne_context"])
    | (scan["n_mne_context"] != scan["n_mne_stimulus"])
]
print(f"Count-level anomalies in {len(anomalies)}/{len(scan)} recordings:")
print(
    anomalies[
        ["file", "n_evt_stimulus", "n_mne_stimulus", "n_dup_stimulus",
         "n_evt_context", "n_mne_context", "n_dup_context"]
    ].to_string(index=False)
)

In [ ]:
order = paired.sort_values("shift_stimulus_s")
fig, axes = plt.subplots(1, 2, figsize=(14, 5), width_ratios=[2, 1])

# Zoomed to the data range: on a 0-based axis every bar looks identical, which is
# the opposite of the point — the spread is what a global constant cannot absorb.
median_ms = np.median(shifts) * 1e3
positions = np.arange(len(order))
axes[0].hlines(positions, median_ms, order["shift_stimulus_s"] * 1e3,
               color="C0", lw=1.2)
axes[0].plot(order["shift_stimulus_s"] * 1e3, positions, "o", ms=4, color="C0")
axes[0].axvline(median_ms, color="C1", lw=1.5, label=f"median {median_ms:.1f} ms")
axes[0].axvline(-PIPELINE_CONSTANT_S * 1e3, color="C3", ls="--", lw=1.5,
                label=f"pipeline constant {-PIPELINE_CONSTANT_S * 1e3:.0f} ms")
axes[0].set_yticks(positions)
axes[0].set_yticklabels(order["file"], fontsize=6)
axes[0].set_ylim(-1, len(order))
axes[0].set(xlabel="shift: MNE − `.evt` (ms)",
            title="Per-recording shift (deviation from the group median)")
axes[0].legend(loc="lower right", fontsize=8, framealpha=0.95)

axes[1].hist(shifts * 1e3, bins=14, color="C0", edgecolor="w")
axes[1].axvline(np.median(shifts) * 1e3, color="C1", lw=1.5)
axes[1].axvline(-PIPELINE_CONSTANT_S * 1e3, color="C3", ls="--", lw=1.5)
axes[1].set(xlabel="shift (ms)", ylabel="recordings", title="Distribution")

fig.suptitle("The shift is a per-recording constant, not a global one", y=1.01)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "shift_across_recordings.png", dpi=150, bbox_inches="tight")
plt.show()

Two things to read off this:

* Every recording has its **own** constant, and they are tightly clustered but not
  identical — a spread of tens of milliseconds around ~425 ms. At 40 Hz one cycle is
  25 ms, so that spread is more than a full cycle of the response being studied.
* `bgin` moves with `fam+` to the nanosecond, so this is a property of the *file*,
  not of any particular trigger code.

So the error is not random, but it is also not a single number — a global constant
cannot remove it. The next step shows it is nonetheless exactly recoverable.

## 4. Mechanism — Two Files, Two Time Origins

The `.evt` export and the EDF measure time from different zeros, and the EDF cannot
describe its own zero precisely enough to reconcile them: **the EDF header stores the
recording start to the nearest whole second**. Any sub-second part of the true start
is dropped when the file is written, and MNE — reading the header faithfully — hands
back annotation times that are late by exactly the dropped remainder.

The `.evt` export is what makes the remainder recoverable: its wall-clock anchor rows
tie one microsecond time to an absolute timestamp. Subtracting the anchor's own
`.evt` time gives the true start of the `.evt` clock in absolute time, and its
distance from the (whole-second) EDF header start is the offset between the origins.

**Prediction.** If this is the mechanism, then for every recording

```
per-recording shift  ==  evt_origin - edf_header_start
```

with nothing left over. The shift was measured from 150 stimulus markers; the origin
offset comes from a single anchor row and the header. They share no input, so
agreement is a real test rather than an algebraic identity.

In [ ]:
paired = paired.assign(
    predicted_s=paired["evt_origin_offset_s"],
    residual_s=paired["shift_stimulus_s"] - paired["evt_origin_offset_s"],
)

print("measured shift vs origin offset predicted from the wall-clock anchor")
print(f"  max |residual| : {paired['residual_s'].abs().max():.3e} s "
      f"(floating-point precision — the two agree exactly)")
print(f"  header start is a whole second in "
      f"{int((paired['header_start'].dt.microsecond == 0).sum())}/{len(paired)} "
      f"recordings (EDF stores no finer resolution)")
print()
print(paired[["file", "header_start", "shift_stimulus_s", "predicted_s", "residual_s"]]
      .head(8).to_string(index=False))

The two agree exactly — to floating-point precision — in every recording. The
discrepancy is therefore **fully explained and fully recoverable**: it is the
sub-second part of the recording start that the EDF header cannot carry, and the
`.evt` anchor hands it back exactly.

That also explains the shape of the distribution in step 3. The offsets are not
scattered uniformly over a whole second, as an arbitrary truncation would be — they
cluster near 425 ms, which is the fixed lead-in the acquisition software records
before the paradigm's own clock starts. The clustering is why a single global
constant looked adequate; the residual spread is why it is not.

## 5. Does the Shift Survive Our Preprocessing?

The question that decides where a fix belongs: is the shift introduced at read time
and then merely carried, or does some pipeline stage add to it? Follow one group
down the chain that ends at the arrays the PCA analyses consume.

Stages differ in what is comparable, so they need different tests.

* `RAW_AFTER_ICA` is still on the original time base, so its annotation times are
  compared to the raw EDF **directly**. Equality here means the shift is inherited,
  not amplified.
* `RAW_CROPPED` is produced by the stimulus aligner, whose whole job is to rewrite
  inter-stimulus intervals. Large numbers in its row are the aligner working, not a
  bug — so the table cannot settle anything for this stage, and the check that
  follows it does: the cropped annotations must reproduce the onset positions
  actually stored alongside the concatenated array.

In [ ]:
group_files = [
    filename
    for filename in group_df[SingleDataMetadata.FILENAME]
    if event_file_path(filename).exists()
]

stage_rows = []
for filename in group_files:
    raw = dataset_handler.load_data_file(filename, preload=False)
    reference = mne_marker_times(raw, STIMULUS_LABEL)
    for variant in STAGE_VARIANTS:
        path = dataset_handler.get_preprocessing_results_path(
            Path(filename).stem, variant
        )
        if not path.exists():
            continue
        staged = dataset_handler.load_data_file(
            filename, is_processed=True, processed_data_type=variant, preload=False
        )
        staged_times = mne_marker_times(staged, STIMULUS_LABEL)
        n = min(len(reference), len(staged_times))
        stage_rows.append(
            {
                "file": Path(filename).stem,
                "stage": variant.value,
                "n_markers": len(staged_times),
                # Absolute marker times vs the raw EDF. Meaningful only while the
                # stage keeps the original time base (RAW_AFTER_ICA).
                "max_abs_change_s": float(
                    np.abs(staged_times[:n] - reference[:n]).max()
                ),
                # Interval structure — invariant to any time origin. The aligner
                # is *supposed* to change this; earlier stages are not.
                "max_interval_change_s": float(
                    np.abs(np.diff(staged_times[:n]) - np.diff(reference[:n])).max()
                )
                if n > 1
                else np.nan,
            }
        )

stages = pd.DataFrame(stage_rows)
summary = stages.groupby("stage").agg(
    recordings=("file", "size"),
    markers=("n_markers", "median"),
    max_abs_change_s=("max_abs_change_s", "max"),
    max_interval_change_s=("max_interval_change_s", "max"),
)
print(summary.to_string())
print()

after_ica_change = summary.loc[
    PreprocessedDataVariants.RAW_AFTER_ICA.value, "max_abs_change_s"
]
verdict = "unchanged" if after_ica_change < STAGE_TOLERANCE_S else "RE-TIMED"
print(f"{PreprocessedDataVariants.RAW_AFTER_ICA.value}: markers {verdict} "
      f"(max {after_ica_change * 1e6:.0f} µs vs {STAGE_TOLERANCE_S * 1e6:.0f} µs "
      f"tolerance for single-precision FIF annotation storage)")
print(f"{PreprocessedDataVariants.RAW_CROPPED.value}: intervals rewritten by the "
      f"aligner as designed — see the check below, not this table")

In [ ]:
onsets_path = (
    dataset_handler.processed_data_dir
    / PreprocessedDataVariants.CONCATENATED.value
    / f"{CONCAT_LABEL}{ProjectPaths.STIMULUS_ONSETS_SUFFIX}"
)
concat_metadata_path = onsets_path.with_name(f"{CONCAT_LABEL}.metadata.csv")

if onsets_path.exists():
    concat_onsets = np.load(onsets_path)
    concat_metadata = pd.read_csv(concat_metadata_path)
    concat_files = [
        Path(name).stem for name in concat_metadata[str(SingleDataMetadata.FILENAME)]
    ]
    print(f"Concatenated array ({CONCAT_LABEL}): "
          f"{len(concat_files)} subjects, {len(concat_onsets)} aligned onsets "
          f"(shared by construction: {concat_onsets[:4]} ... {concat_onsets[-2:]})")

    # Do the cropped annotations still explain the stored onsets? Read them the way
    # every analysis does — through `get_stimulus_onset_samples` with the pipeline's
    # offset — and map onto the concatenated sample grid.
    onset_deviations = []
    for filename in concat_metadata[str(SingleDataMetadata.FILENAME)]:
        cropped = dataset_handler.load_data_file(
            filename,
            is_processed=True,
            processed_data_type=PreprocessedDataVariants.RAW_CROPPED,
            preload=False,
        )
        samples = get_stimulus_onset_samples(
            cropped, STIMULUS_LABEL, PIPELINE_CONSTANT_S
        )
        rescaled = np.round(samples * CONCAT_SFREQ / cropped.info["sfreq"]).astype(int)
        n = min(len(rescaled), len(concat_onsets))
        onset_deviations.append(int(np.abs(rescaled[:n] - concat_onsets[:n]).max()))

    worst = max(onset_deviations)
    print(f"  cropped annotations vs stored onsets: max deviation {worst} sample(s) "
          f"= {worst / CONCAT_SFREQ * 1e3:.0f} ms (resampling rounding)")
    print()

    concat_shifts = paired.set_index("file")["shift_stimulus_s"].reindex(concat_files)
    print("Every subject is cut at the SAME sample, but each one's true stimulus")
    print("onset is its own shift earlier than its marker. Residual timing error at")
    print("the input of the PCA / evoked analyses:")
    residual_at_pca = concat_shifts.to_numpy() + PIPELINE_CONSTANT_S
    print(f"  {np.nanmin(residual_at_pca) * 1e3:+.1f} ms .. "
          f"{np.nanmax(residual_at_pca) * 1e3:+.1f} ms   "
          f"(median {np.nanmedian(residual_at_pca) * 1e3:+.1f} ms, "
          f"spread {np.ptp(residual_at_pca[~np.isnan(residual_at_pca)]) * 1e3:.1f} ms)")
else:
    residual_at_pca = None
    print(f"No concatenated product at {onsets_path} — skipping.")

`RAW_AFTER_ICA` reproduces the raw annotation times to ~14 µs — the resolution at
which FIF stores annotation onsets, i.e. unchanged. And the cropped annotations still
land on the stored concatenated onsets to within one resampled sample. **Our
preprocessing adds nothing**: it inherits the read-time shift and passes it on, and
the stimulus-aligned products downstream are cut around markers that were already
late when they were read.

The fix therefore belongs at the point where annotations are read — not anywhere in
the pipeline.

## 6. Consequences for `AssrEpoch.MARKER_ONSET_OFFSET_S`

The pipeline currently applies one global constant, `-0.400 s`, measured by ITC. The
scan says the true correction is a per-recording constant. Comparing the two shows
what the global constant leaves on the table: a per-subject residual that no amount
of averaging removes, because it is a *fixed* mis-timing per subject, not noise.

In [ ]:
global_residual = shifts + PIPELINE_CONSTANT_S  # what remains today
cycle_ms = 1e3 / 40.0
print(f"Global constant {PIPELINE_CONSTANT_S:+.3f} s")
print(f"  residual per recording : {global_residual.min() * 1e3:+.1f} .. "
      f"{global_residual.max() * 1e3:+.1f} ms "
      f"(median {np.median(global_residual) * 1e3:+.1f} ms)")
print(f"  spread                 : {np.ptp(global_residual) * 1e3:.1f} ms "
      f"= {np.ptp(global_residual) * 1e3 / cycle_ms:.1f} cycles of the 40 Hz response")
print(f"  |residual| > half a cycle in "
      f"{int((np.abs(global_residual) * 1e3 > cycle_ms / 2).sum())}/{len(shifts)} "
      f"recordings")
print()
print("Per-recording correction: residual is zero by construction — step 4 showed")
print("the shift is reproduced exactly from the `.evt` anchor, so subtracting it")
print("leaves nothing. The comparison is global-constant vs no error at all.")
print()
print(f"Best single constant available (median of the measured shifts): "
      f"{-np.median(shifts):+.4f} s "
      f"(would still leave {np.ptp(shifts) * 1e3:.0f} ms of spread)")

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(global_residual * 1e3, bins=14, color="C3", alpha=0.8, edgecolor="w",
        label=f"global {PIPELINE_CONSTANT_S:+.3f} s")
ax.axvline(0.0, color="k", lw=1.2)
for edge in (-cycle_ms / 2, cycle_ms / 2):
    ax.axvline(edge, color="C0", ls=":", lw=1.2)
ax.text(cycle_ms / 2, ax.get_ylim()[1] * 0.95, "  ±½ cycle @ 40 Hz",
        color="C0", fontsize=9, va="top")
ax.set(
    xlabel="residual onset error per recording (ms)",
    ylabel="recordings",
    title="What the global constant leaves behind\n"
          "(a per-recording correction leaves nothing)",
)
ax.legend()
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "residual_after_correction.png", dpi=150,
                bbox_inches="tight")
plt.show()

## Conclusion

**The discrepancy is systematic, not random — and exactly recoverable.**

| Level | Behaviour |
|---|---|
| Within a recording | One constant shift. Residual ≤ 2 µs over ~150 markers and ~4 min — no drift, no per-event jitter. |
| Across recordings | A *per-recording* constant, 372–455 ms (median 425 ms). Identical for `fam+` and `bgin`, so it belongs to the file, not the trigger code. |
| Cause | The EDF header records the recording start only to the nearest whole second; the `.evt` export uses its own origin, which sits ~425 ms later. MNE reads the header faithfully, so every annotation comes back late by the dropped remainder. |
| Recoverability | Exact. `shift = evt_wall_clock_origin − edf_header_start`, matching the measured shift to floating-point precision in all 38 recordings. |
| Our pipeline | Adds nothing. `RAW_AFTER_ICA` reproduces the raw annotation times to the µs; the cropped annotations still explain the stored concatenated onsets. Downstream stages inherit the error, they do not create it. |

**Marker counts are intact.** `fam+` counts agree between the two sources in every
recording — nothing is dropped, gained or reordered. Three export-side quirks show
up in the anomaly table; none of them is a timing problem, and none of them touches
`fam+` timing:

* One export (`PSI030_EEGB`) lists every `fam+` twice, once per hardware trigger
  line, at an identical microsecond — hence the de-duplication in the helper.
* Several recordings carry a `bgin` with no matching `fam+` (150 vs 149) — the
  `orphan_bgin` exclusion already on record.
* One export (`PSI028_EEGB`) lists 138 `bgin` where the EDF has 150. This is a
  `.evt`-side gap in the *companion* marker only; its `fam+` markers are complete
  and match the EDF exactly, so its shift is still measurable.

**What this means for the analyses.** The global `-0.4` s constant was a
good estimate of the median (measured shift: −0.425 s) and it removes most of the
error. What it cannot remove is the per-recording spread: after the global
correction each subject is still mis-timed by a *fixed* amount spanning ~83 ms,
which is more than three cycles of the 40 Hz response. That is not averaged away by
pooling subjects — it directly de-synchronises any cross-subject phase-locked
measure (ITC, evoked averaging, the channel-PCA evoked input).

## Status — the fix has landed

This diagnosis has been acted on; the notebook is kept as the evidence for *why*,
and it deliberately still measures against the old global `-0.4` s
(`PIPELINE_CONSTANT_S` in the configuration cell) so that the case for replacing it
stays reproducible.

What is in the pipeline now:

* `scripts/run_stimulus_shift_calibration.py` measures every recording against its
  `.evt` export — using both estimators shown above, and refusing to write when they
  disagree — and stores the result in
  `config/participant_mappings/assr_time_shift.csv`.
* `resolve_stimulus_marker(experiment_name)` in
  `src/preprocessing/stimulus_alignment.py` is the single place that resolves an
  offset. `StimulusMarker.onset_offset_for(filename)` returns one recording's value;
  `AssrEpoch.MARKER_ONSET_OFFSET_S` is now only the fallback for recordings the
  calibration does not cover (set to the median of the measured shifts).
* The coarse crop, the group alignment (`align_raws` accepts one offset per
  recording), and the concatenated-onset extraction all read offsets that way, so
  recordings are aligned on the **stimulus** rather than on their markers.

Re-running the calibration invalidates **every** product from the coarse crop
onwards — including preprocessing. The stimulus-aware coarse crop trims around the
*offset-adjusted* onsets, so recordings cropped under the old constant carry too
little lead-in for the new offsets (measured: 49 ms before the first true onset,
against the 500 ms a rerun keeps and the 100 ms `AssrEpoch.PRE_ONSET_S` assumes).
`StimulusAligner` takes `pre_target` as the group minimum, so one under-cropped
recording would shorten the pre-stimulus baseline for the whole group, silently.

Regenerate in this order: `RAW_BEFORE_ICA`/`RAW_AFTER_ICA` (re-run preprocessing),
`RAW_CROPPED`, the concatenated array with its `.stimulus_onsets.npy`, and the
wavelet cache.